In [ ]:
# import hydrobr

In [ ]:
# estacoes = ["2146117"]

# # Para chuva/precipitação
# #dados_chuva = hydrobr.get_data.ANA.prec_data(estacoes)

# print("--- Dados de Vazão ---")
# # Para vazão
# dados_vazao = hydrobr.get_data.ANA.flow_data(estacoes)

# # Ver quantidade total de linhas baixadas
# print(f"Total de dias com registro de vazão: {len(dados_vazao)}")

# # Preencher pequenos vazios ou remover linhas vazias
# dados_vazao_limpo = dados_vazao.dropna()

# print("\n--- Dados de Chuva ---")
# #print(dados_chuva.head())

--- Dados de Vazão ---


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


ValueError: No objects to concatenate

In [6]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET


def obter_telemetria_recente(cod_estacao, data_inicio="01/01/2019"):
    """Puxa dados telemétricos (brutos) recentes diretamente da API de telemetria da ANA."""
    # Código da estação com 8 dígitos
    cod_estacao_str = str(cod_estacao).zfill(8)

    url = (
        f"http://telemetriaws1.ana.gov.br/ServiceANA.asmx/DadosHidrometeorologicos?"
        f"codEstacao={cod_estacao_str}&dataInicio={data_inicio}&dataFim="
    )

    print(f"Buscando telemetria recente da estação {cod_estacao_str}...")
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro na requisição: Status {response.status_code}")
        return pd.DataFrame()

    root = ET.fromstring(response.content)

    registros = []
    for dados in root.findall(".//DadosHidrometereologicos"):
        data_hora = (
            dados.find("DataHora").text
            if dados.find("DataHora") is not None
            else None
        )
        vazao = (
            dados.find("Vazao").text
            if dados.find("Vazao") is not None
            else None
        )
        chuva = (
            dados.find("Chuva").text
            if dados.find("Chuva") is not None
            else None
        )
        cota = (
            dados.find("Nivel").text if dados.find("Nivel") is not None else None
        )

        if data_hora:
            registros.append(
                {
                    "DataHora": pd.to_datetime(data_hora),
                    "Vazao": float(vazao) if vazao else None,
                    "Nivel": float(cota) if cota else None,
                    "Chuva": float(chuva) if chuva else None,
                }
            )

    df = pd.DataFrame(registros)
    if not df.empty:
        df = df.sort_values(by="DataHora").reset_index(drop=True)
    return df


# Executando para a estação 61807002
df_recente = obter_telemetria_recente("2146117", data_inicio="01/01/2019")
print(f"Total de registros recentes encontrados: {len(df_recente)}")
print(df_recente.tail(10))

Buscando telemetria recente da estação 02146117...
Total de registros recentes encontrados: 0
Empty DataFrame
Columns: []
Index: []


In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import requests


def obter_telemetria_diaria(
    cod_estacao, data_inicio="01/01/2020", apenas_aprovados=True
):
    """Puxa dados telemétricos da ANA, filtra por consistência (aprovados)

    e calcula os agregados diários (Média para Vazão/Nivel, Soma para Chuva).
    """
    cod_estacao_str = str(cod_estacao).zfill(8)

    url = (
        f"http://telemetriaws1.ana.gov.br/ServiceANA.asmx/DadosHidrometeorologicos?"
        f"codEstacao={cod_estacao_str}&dataInicio={data_inicio}&dataFim="
    )

    print(f"Buscando dados da estação {cod_estacao_str}...")
    try:
        response = requests.get(url, timeout=30)
    except Exception as e:
        print(f"Erro de conexão com a estação {cod_estacao_str}: {e}")
        return pd.DataFrame()

    if response.status_code != 200:
        print(f"Erro na requisição: Status {response.status_code}")
        return pd.DataFrame()

    root = ET.fromstring(response.content)

    registros = []
    for dados in root.findall(".//DadosHidrometereologicos"):
        data_hora = (
            dados.find("DataHora").text
            if dados.find("DataHora") is not None
            else None
        )
        vazao = (
            dados.find("Vazao").text
            if dados.find("Vazao") is not None
            else None
        )
        chuva = (
            dados.find("Chuva").text
            if dados.find("Chuva") is not None
            else None
        )
        cota = (
            dados.find("Nivel").text if dados.find("Nivel") is not None else None
        )

        # NivelConsistencia: 1 = Bruto, 2 = Aprovado/Consistido
        consistencia = (
            dados.find("NivelConsistencia").text
            if dados.find("NivelConsistencia") is not None
            else "1"
        )

        if data_hora:
            registros.append(
                {
                    "DataHora": pd.to_datetime(data_hora),
                    "Vazao": float(vazao) if vazao else None,
                    "Nivel": float(cota) if cota else None,
                    "Chuva": float(chuva) if chuva else None,
                    "NivelConsistencia": (
                        int(consistencia) if consistencia else 1
                    ),
                }
            )

    df = pd.DataFrame(registros)
    if df.empty:
        print(f"Nenhum dado retornado para a estação {cod_estacao_str}.")
        return pd.DataFrame()

    # 1. Filtro de dados aprovados (NivelConsistencia == 2)
    if apenas_aprovados:
        df_aprovado = df[df["NivelConsistencia"] == 2]
        # Se a telemetria recente ainda não foi consistida, avisa o usuário
        if df_aprovado.empty:
            print(
                f"[Aviso] Nenhum dado com status 'Aprovado' (Nível 2) encontrado para {cod_estacao_str}. Retornando dados brutos disponíveis."
            )
        else:
            df = df_aprovado

    # 2. Agregação Diária
    df["Data"] = df["DataHora"].dt.date
    df_diario = (
        df.groupby("Data")
        .agg(
            {
                "Vazao": "mean",  # Média diária da vazão
                "Nivel": "mean",  # Média diária do nível (cota)
                "Chuva": "sum",  # Soma diária da precipitação
            }
        )
        .reset_index()
    )

    df_diario["CodEstacao"] = cod_estacao_str
    df_diario = df_diario.sort_values(by="Data").reset_index(drop=True)

    return df_diario


# --- EXECUÇÃO PARA AS ESTAÇÕES DESEJADAS ---
estacoes = ["2146117"] #"61807002", "61811080", "61802502"
lista_dfs = []

for estacao in estacoes:
    df_estacao = obter_telemetria_diaria(
        estacao, data_inicio="01/01/2020", apenas_aprovados=True
    )
    if not df_estacao.empty:
        lista_dfs.append(df_estacao)

# Junta todos os dados em um único DataFrame consolidado
if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)

    # Reorganiza as colunas
    df_final = df_final[
        ["CodEstacao", "Data", "Vazao", "Nivel", "Chuva"]
    ].round(2)

    print("\n--- RESUMO CONSOLIDADO ---")
    print(df_final.head(15))

    # Salva em CSV
    df_final.to_csv(f"{estacao}_telemetria_diaria_estacoes_nivel.csv", index=False, sep=";")
    print(
        "\nArquivo salvo com sucesso: 'telemetria_diaria_estacoes_nivel.csv' (delimitador ';')"
    )

Buscando dados da estação 02146117...
Nenhum dado retornado para a estação 02146117.
